In [0]:
%sql
-- Databricks notebook source
-- Sales per day and store

CREATE OR REPLACE MATERIALIZED VIEW workspace.default.mv_ventas_tienda_dia (
  store,
  date,
  ordenes,
  unidades,
  ventas,
  ticket_promedio,
  CONSTRAINT tienda_y_fecha EXPECT (store IS NOT NULL AND date IS NOT NULL) ON VIOLATION FAIL UPDATE,
  CONSTRAINT ventas_no_negativas EXPECT (ventas >= 0)
)
CLUSTER BY (store, date)
AS
WITH por_orden AS (
  SELECT store,
         date,
         check_number,
         MAX(total) AS total_orden,
         COUNT(*)   AS unidades
  FROM workspace.default.gold_ventas
  GROUP BY store, date, check_number
)
SELECT store,
       date,
       COUNT(*)                              AS ordenes,
       SUM(unidades)                         AS unidades,
       SUM(total_orden)                      AS ventas,
       ROUND(SUM(total_orden) / COUNT(*), 2) AS ticket_promedio
FROM por_orden
GROUP BY store, date;